In [0]:
import pandas as pd

# 1. Définition de l'URL du fichier CSV
url = "https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/products.csv"

# 2. Chargement des données dans un DataFrame Pandas (exécuté sur le driver)
pdf = pd.read_csv(url)

# 3. Conversion du DataFrame Pandas en DataFrame Spark (distribué sur le cluster)
df = spark.createDataFrame(pdf)

# 4. Affichage des 10 premières lignes dans l'interface Databricks
display(df.limit(10))

ProductID,ProductName,Category,ListPrice
771,"Mountain-100 Silver, 38",Mountain Bikes,3399.99
772,"Mountain-100 Silver, 42",Mountain Bikes,3399.99
773,"Mountain-100 Silver, 44",Mountain Bikes,3399.99
774,"Mountain-100 Silver, 48",Mountain Bikes,3399.99
775,"Mountain-100 Black, 38",Mountain Bikes,3374.99
776,"Mountain-100 Black, 42",Mountain Bikes,3374.99
777,"Mountain-100 Black, 44",Mountain Bikes,3374.99
778,"Mountain-100 Black, 48",Mountain Bikes,3374.99
779,"Mountain-200 Silver, 38",Mountain Bikes,2319.99
780,"Mountain-200 Silver, 42",Mountain Bikes,2319.99


In [0]:
# 1. On définit un nom de table à la place d'un chemin
# Remarque : 'default' est la base de données par défaut, 'products_delta' le nom de votre table
table_name = "default.products_delta"

# 2. On utilise 'saveAsTable' au lieu de 'save'
# Cela enregistre les données dans le stockage géré sécurisé de Databricks
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

# 3. Message de confirmation
print(f"Table Delta enregistrée avec succès dans le catalogue sous le nom : {table_name}")

# 4. Pour voir le résultat, on lit la table
display(spark.table(table_name))

Table Delta enregistrée avec succès dans le catalogue sous le nom : default.products_delta


ProductID,ProductName,Category,ListPrice
771,"Mountain-100 Silver, 38",Mountain Bikes,3399.99
772,"Mountain-100 Silver, 42",Mountain Bikes,3399.99
773,"Mountain-100 Silver, 44",Mountain Bikes,3399.99
774,"Mountain-100 Silver, 48",Mountain Bikes,3399.99
775,"Mountain-100 Black, 38",Mountain Bikes,3374.99
776,"Mountain-100 Black, 42",Mountain Bikes,3374.99
777,"Mountain-100 Black, 44",Mountain Bikes,3374.99
778,"Mountain-100 Black, 48",Mountain Bikes,3374.99
779,"Mountain-200 Silver, 38",Mountain Bikes,2319.99
780,"Mountain-200 Silver, 42",Mountain Bikes,2319.99


In [0]:
from delta.tables import *
from pyspark.sql.functions import *

# 1. Charger la table en utilisant son NOM (et non son chemin)
# Nous utilisons la table 'default.products_delta' créée à l'étape précédente
deltaTable = DeltaTable.forName(spark, "default.products_delta")

# 2. Mettre à jour le prix (réduction de 10%) pour le produit 771
deltaTable.update(
    condition = "ProductID == 771",
    set = { "ListPrice": "ListPrice * 0.9" }
)

# 3. Afficher les résultats
# toDF() convertit l'objet DeltaTable en DataFrame Spark classique pour l'affichage
display(deltaTable.toDF().limit(10))

ProductID,ProductName,Category,ListPrice
772,"Mountain-100 Silver, 42",Mountain Bikes,3399.99
773,"Mountain-100 Silver, 44",Mountain Bikes,3399.99
774,"Mountain-100 Silver, 48",Mountain Bikes,3399.99
775,"Mountain-100 Black, 38",Mountain Bikes,3374.99
776,"Mountain-100 Black, 42",Mountain Bikes,3374.99
777,"Mountain-100 Black, 44",Mountain Bikes,3374.99
778,"Mountain-100 Black, 48",Mountain Bikes,3374.99
779,"Mountain-200 Silver, 38",Mountain Bikes,2319.99
780,"Mountain-200 Silver, 42",Mountain Bikes,2319.99
781,"Mountain-200 Silver, 46",Mountain Bikes,2319.99


In [0]:
# 1. Chargement des données à partir du nom de la table
# On utilise le nom utilisé précédemment : "default.products_delta"
new_df = spark.read.table("default.products_delta")

# 2. Affichage des 10 premières lignes
# display() est plus lisible dans Databricks, mais .show(10) fonctionne aussi
display(new_df.limit(10))

ProductID,ProductName,Category,ListPrice
772,"Mountain-100 Silver, 42",Mountain Bikes,3399.99
773,"Mountain-100 Silver, 44",Mountain Bikes,3399.99
774,"Mountain-100 Silver, 48",Mountain Bikes,3399.99
775,"Mountain-100 Black, 38",Mountain Bikes,3374.99
776,"Mountain-100 Black, 42",Mountain Bikes,3374.99
777,"Mountain-100 Black, 44",Mountain Bikes,3374.99
778,"Mountain-100 Black, 48",Mountain Bikes,3374.99
779,"Mountain-200 Silver, 38",Mountain Bikes,2319.99
780,"Mountain-200 Silver, 42",Mountain Bikes,2319.99
781,"Mountain-200 Silver, 46",Mountain Bikes,2319.99


In [0]:
# On récupère l'historique de la table par son nom
deltaTable = DeltaTable.forName(spark, "default.products_delta")
display(deltaTable.history(10))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-04-28T12:33:36.000Z,73569034942853,a.driss@adgeducation.com,UPDATE,"Map(predicate -> [""(ProductID#12480L = 771)""])",null,List(2855742434779946),2a60ad8d-698f-45e9-b2f9-b445b45b92da,0428-121325-6o1120ky-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4025, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1858, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1488, rewriteTimeMs -> 2120)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
2,2026-04-28T12:33:29.000Z,73569034942853,a.driss@adgeducation.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2855742434779946),4b3d56ad-7cee-4d51-ba38-9103d7514f3c,0428-121325-6o1120ky-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 5849, numDeletionVectorsRemoved -> 1, numOutputRows -> 295, numOutputBytes -> 4361)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-04-28T10:38:29.000Z,73569034942853,a.driss@adgeducation.com,UPDATE,"Map(predicate -> [""(ProductID#11372L = 771)""])",null,List(2855742434779946),f63dfc2d-83ef-4d35-bbc1-3ccd9ed681ab,0428-102301-q66av0fy-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4611, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2169, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1488, rewriteTimeMs -> 2407)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-04-28T10:36:53.000Z,73569034942853,a.driss@adgeducation.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2855742434779946),56e04d76-05eb-4b7d-82f3-0538cf1d0c3c,0428-102301-q66av0fy-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 295, numOutputBytes -> 4361)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


In [0]:
# 1. Création de la base de données (Schéma)
spark.sql("CREATE DATABASE IF NOT EXISTS AdventureWorks")

# 2. Création de la table externe
# NOTE : On retire 'hive_metastore' pour laisser Databricks gérer le catalogue par défaut (Unity Catalog)
# On utilise le format Delta sans spécifier de LOCATION bloquée pour laisser le stockage géré agir
spark.sql("""
CREATE TABLE IF NOT EXISTS AdventureWorks.ProductsExternal
USING DELTA
""")

# 3. Affichage des détails de la table
# Cela permet de voir où Databricks a physiquement stocké les données
spark.sql("""
DESCRIBE EXTENDED AdventureWorks.ProductsExternal
""").show(truncate=False)

+----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                                                                                                                                                                                                                                                                                                     

In [0]:
%sql
-- 1. Supprimer la table vide
DROP TABLE IF EXISTS AdventureWorks.ProductsExternal;

-- 2. Créer la table avec les données de products_delta
CREATE TABLE AdventureWorks.ProductsExternal
USING DELTA
AS SELECT * FROM default.products_delta;

-- 3. Lire le contenu de la table
SELECT * FROM AdventureWorks.ProductsExternal;

ProductID,ProductName,Category,ListPrice
771,"Mountain-100 Silver, 38",Mountain Bikes,3059.991
772,"Mountain-100 Silver, 42",Mountain Bikes,3399.99
773,"Mountain-100 Silver, 44",Mountain Bikes,3399.99
774,"Mountain-100 Silver, 48",Mountain Bikes,3399.99
775,"Mountain-100 Black, 38",Mountain Bikes,3374.99
776,"Mountain-100 Black, 42",Mountain Bikes,3374.99
777,"Mountain-100 Black, 44",Mountain Bikes,3374.99
778,"Mountain-100 Black, 48",Mountain Bikes,3374.99
779,"Mountain-200 Silver, 38",Mountain Bikes,2319.99
780,"Mountain-200 Silver, 42",Mountain Bikes,2319.99


In [0]:
# 1. Enregistrement du DataFrame en tant que table gérée dans le catalogue
# On utilise le format Delta et le mode "overwrite" pour écraser si elle existe déjà
df.write.format("delta").mode("overwrite").saveAsTable("AdventureWorks.ProductsManaged")

# 2. Affichage des métadonnées de la table
# Cela permet de confirmer que c'est une table MANAGED et de voir son emplacement réel
spark.sql("DESCRIBE EXTENDED AdventureWorks.ProductsManaged").show(truncate=False)

+----------------------------+-------------------------------------------+-------+
|col_name                    |data_type                                  |comment|
+----------------------------+-------------------------------------------+-------+
|ProductID                   |bigint                                     |NULL   |
|ProductName                 |string                                     |NULL   |
|Category                    |string                                     |NULL   |
|ListPrice                   |double                                     |NULL   |
|                            |                                           |       |
|# Delta Statistics Columns  |                                           |       |
|Column Names                |ProductID, ProductName, Category, ListPrice|       |
|Column Selection Method     |first-32                                   |       |
|                            |                                           |       |
|# D

In [0]:
%sql
-- Sélection de la base de données
USE AdventureWorks;

-- Lecture des données de la table gérée
SELECT * FROM ProductsManaged;

ProductID,ProductName,Category,ListPrice
771,"Mountain-100 Silver, 38",Mountain Bikes,3399.99
772,"Mountain-100 Silver, 42",Mountain Bikes,3399.99
773,"Mountain-100 Silver, 44",Mountain Bikes,3399.99
774,"Mountain-100 Silver, 48",Mountain Bikes,3399.99
775,"Mountain-100 Black, 38",Mountain Bikes,3374.99
776,"Mountain-100 Black, 42",Mountain Bikes,3374.99
777,"Mountain-100 Black, 44",Mountain Bikes,3374.99
778,"Mountain-100 Black, 48",Mountain Bikes,3374.99
779,"Mountain-200 Silver, 38",Mountain Bikes,2319.99
780,"Mountain-200 Silver, 42",Mountain Bikes,2319.99


In [0]:
%sql
USE AdventureWorks;
SHOW TABLES;

database,tableName,isTemporary
adventureworks,productsexternal,false
adventureworks,productsmanaged,false


In [0]:
# Inspection des métadonnées des tables Delta
# Cette approche fonctionne avec Unity Catalog sans nécessiter d'accès direct aux fichiers

print("=== Détails de la table Externe ===")
display(spark.sql("DESCRIBE DETAIL AdventureWorks.ProductsExternal"))

print("\n=== Détails de la table Gérée ===")
display(spark.sql("DESCRIBE DETAIL AdventureWorks.ProductsManaged"))

print("\n=== Historique des versions (ProductsExternal) ===")
display(spark.sql("DESCRIBE HISTORY AdventureWorks.ProductsExternal LIMIT 5"))

=== Détails de la table Externe ===


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,a99a7057-db5e-43c3-a202-699184b45fd0,workspace.adventureworks.productsexternal,null,,2026-04-28T12:33:44.443Z,2026-04-28T12:33:46.000Z,List(),List(),1,4367,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false



=== Détails de la table Gérée ===


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,1ff01dfa-90ca-4da7-b9c5-a9672f7681e0,workspace.adventureworks.productsmanaged,null,,2026-04-28T12:33:49.040Z,2026-04-28T12:33:51.000Z,List(),List(),1,4361,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false



=== Historique des versions (ProductsExternal) ===


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-04-28T12:33:46.000Z,73569034942853,a.driss@adgeducation.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2855742434779946),91fcb11f-9d9a-4900-9c93-3f90a94fbf5d,0428-121325-6o1120ky-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 295, numOutputBytes -> 4367)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


In [0]:
%sql
USE AdventureWorks;

DROP TABLE IF EXISTS ProductsExternal;
DROP TABLE IF EXISTS ProductsManaged;

SHOW TABLES;

database,tableName,isTemporary


In [0]:
%sql
    
-- 1. S'assurer d'utiliser la bonne base de données
USE AdventureWorks;

-- 2. Recréer la table à partir des données existantes
-- Unity Catalog ne permet pas LOCATION 'dbfs:/...', donc on utilise CREATE TABLE AS SELECT
CREATE TABLE IF NOT EXISTS Products
USING DELTA
AS SELECT * FROM default.products_delta;

-- 3. Vérifier que les données sont bien présentes
SELECT * FROM Products LIMIT 10;

ProductID,ProductName,Category,ListPrice
771,"Mountain-100 Silver, 38",Mountain Bikes,3059.991
772,"Mountain-100 Silver, 42",Mountain Bikes,3399.99
773,"Mountain-100 Silver, 44",Mountain Bikes,3399.99
774,"Mountain-100 Silver, 48",Mountain Bikes,3399.99
775,"Mountain-100 Black, 38",Mountain Bikes,3374.99
776,"Mountain-100 Black, 42",Mountain Bikes,3374.99
777,"Mountain-100 Black, 44",Mountain Bikes,3374.99
778,"Mountain-100 Black, 48",Mountain Bikes,3374.99
779,"Mountain-200 Silver, 38",Mountain Bikes,2319.99
780,"Mountain-200 Silver, 42",Mountain Bikes,2319.99


In [0]:
# 1. Consolider les petits fichiers (Compaction)
# La commande OPTIMIZE regroupe les fichiers fragmentés pour améliorer les performances.
spark.sql("OPTIMIZE Products")

# 2. Nettoyer les anciens fichiers (VACUUM)
# Supprime les fichiers qui ne sont plus référencés dans la version actuelle 
# et qui sont plus vieux que le seuil spécifié (minimum 168 heures = 7 jours).
spark.sql("VACUUM Products RETAIN 168 HOURS")

DataFrame[path: string]